# Semana 14: Integrales y Aplicaciones en Evaluación de Modelos
## Notebook de Ejercicios (NB2) - Dataset Titanic

### Propósito de la sesión
Aplicar los conceptos de **integral definida** y **AUC** a un problema real de clasificación binaria. Trabajaremos con el dataset **Titanic**, entrenaremos diferentes clasificadores, calcularemos la curva ROC y el AUC manualmente (mediante integración trapezoidal) y con `sklearn`, y compararemos el rendimiento de varios modelos.

### Objetivos de aprendizaje
*   Cargar y preprocesar el dataset **Titanic**.
*   Entrenar diferentes clasificadores (regresión logística, árbol de decisión, etc.).
*   Calcular la **curva ROC** para cada modelo.
*   Calcular el **AUC** manualmente usando integración trapezoidal (`np.trapz`).
*   Comparar con `sklearn.metrics.roc_auc_score`.
*   Comparar el AUC de diferentes modelos y discutir cuál es mejor.

## Configuración Inicial

Importamos las librerías necesarias: pandas para manipulación, numpy, matplotlib, seaborn para visualización, y sklearn para modelos y métricas.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, roc_auc_score, auc

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)

# Fijamos semilla para reproducibilidad
np.random.seed(42)

print("🎯 Librerías importadas correctamente.")

---
## 1. Carga y Exploración del Dataset Titanic

Cargamos el dataset Titanic desde un URL o desde seaborn. Usaremos el dataset incluido en seaborn por simplicidad.

In [ ]:
# Cargamos el dataset Titanic desde seaborn
titanic = sns.load_dataset('titanic')

print("🔷 Dataset Titanic cargado:")
print(f"  Shape: {titanic.shape}")
print(f"  Columnas: {titanic.columns.tolist()}")
display(titanic.head())

In [ ]:
# Información general
print("📊 Información del dataset:")
titanic.info()

# Estadísticas descriptivas
display(titanic.describe(include='all'))

### 1.1. Preprocesamiento básico

Seleccionamos características relevantes y manejamos valores faltantes.

In [ ]:
# Seleccionamos características
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
X = titanic[features].copy()
y = titanic['survived'].copy()

# Codificamos variables categóricas
X['sex'] = LabelEncoder().fit_transform(X['sex'])  # male=1, female=0
X = pd.get_dummies(X, columns=['embarked'], prefix='emb', drop_first=True)

# Imputamos valores faltantes
imputer_age = SimpleImputer(strategy='median')
X['age'] = imputer_age.fit_transform(X[['age']])

imputer_fare = SimpleImputer(strategy='median')
X['fare'] = imputer_fare.fit_transform(X[['fare']])

print("🔶 Features después del preprocesamiento:")
print(f"  Shape: {X.shape}")
print(f"  Columnas: {X.columns.tolist()}")
display(X.head())

In [ ]:
# Dividimos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Estandarizamos características numéricas
scaler = StandardScaler()
num_cols = ['age', 'sibsp', 'parch', 'fare']
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(f"Shape X_train: {X_train.shape}")
print(f"Shape X_test: {X_test.shape}")

---
## Actividad 1: Entrenar clasificadores

Entrenamos tres modelos diferentes: regresión logística, árbol de decisión y random forest.

In [ ]:
# 1. Regresión Logística
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)
y_scores_log = log_reg.predict_proba(X_test)[:, 1]

# 2. Árbol de Decisión
tree_clf = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_clf.fit(X_train, y_train)
y_scores_tree = tree_clf.predict_proba(X_test)[:, 1]

# 3. Random Forest
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_clf.fit(X_train, y_train)
y_scores_rf = rf_clf.predict_proba(X_test)[:, 1]

print("✅ Modelos entrenados.")

---
## Actividad 2: Calcular curva ROC y AUC manualmente

### 2.1. Función para calcular TPR y FPR para diferentes umbrales

In [ ]:
def roc_manual(y_true, y_scores, num_thresholds=100):
    """
    Calcula TPR y FPR para diferentes umbrales.
    """
    thresholds = np.linspace(0, 1, num_thresholds)
    tpr_list = []
    fpr_list = []
    
    for thresh in thresholds:
        y_pred = (y_scores >= thresh).astype(int)
        tp = np.sum((y_pred == 1) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        fn = np.sum((y_pred == 0) & (y_true == 1))
        tn = np.sum((y_pred == 0) & (y_true == 0))
        
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        
        tpr_list.append(tpr)
        fpr_list.append(fpr)
    
    # Ordenamos por FPR (para integración)
    idx = np.argsort(fpr_list)
    return np.array(fpr_list)[idx], np.array(tpr_list)[idx]

# Calculamos ROC manual para cada modelo
fpr_log, tpr_log = roc_manual(y_test.values, y_scores_log)
fpr_tree, tpr_tree = roc_manual(y_test.values, y_scores_tree)
fpr_rf, tpr_rf = roc_manual(y_test.values, y_scores_rf)

### 2.2. Calcular AUC manualmente usando integración trapezoidal

In [ ]:
def auc_manual(fpr, tpr):
    """Calcula AUC usando la regla trapezoidal."""
    return np.trapz(tpr, fpr)

auc_log_manual = auc_manual(fpr_log, tpr_log)
auc_tree_manual = auc_manual(fpr_tree, tpr_tree)
auc_rf_manual = auc_manual(fpr_rf, tpr_rf)

print("🔷 AUC calculado manualmente (trapz):")
print(f"  Regresión Logística: {auc_log_manual:.4f}")
print(f"  Árbol de Decisión:   {auc_tree_manual:.4f}")
print(f"  Random Forest:        {auc_rf_manual:.4f}")

### 2.3. Comparar con `sklearn.metrics.roc_auc_score`

In [ ]:
auc_log_sk = roc_auc_score(y_test, y_scores_log)
auc_tree_sk = roc_auc_score(y_test, y_scores_tree)
auc_rf_sk = roc_auc_score(y_test, y_scores_rf)

print("🔶 AUC calculado con sklearn:")
print(f"  Regresión Logística: {auc_log_sk:.4f}")
print(f"  Árbol de Decisión:   {auc_tree_sk:.4f}")
print(f"  Random Forest:        {auc_rf_sk:.4f}")

# Verificamos diferencias
print("\n📊 Comparación manual vs sklearn:")
print(f"  Logística: diff = {abs(auc_log_manual - auc_log_sk):.2e}")
print(f"  Árbol:     diff = {abs(auc_tree_manual - auc_tree_sk):.2e}")
print(f"  RF:        diff = {abs(auc_rf_manual - auc_rf_sk):.2e}")

---
## Actividad 3: Comparar el AUC de diferentes modelos

Visualizamos las curvas ROC de los tres modelos en una misma gráfica para comparar.

In [ ]:
plt.figure(figsize=(10, 8))

plt.plot(fpr_log, tpr_log, 'b-', linewidth=2, label=f'Logistic Regression (AUC={auc_log_sk:.4f})')
plt.plot(fpr_tree, tpr_tree, 'g-', linewidth=2, label=f'Decision Tree (AUC={auc_tree_sk:.4f})')
plt.plot(fpr_rf, tpr_rf, 'r-', linewidth=2, label=f'Random Forest (AUC={auc_rf_sk:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatorio (AUC=0.5)')

plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Comparación de Curvas ROC en Titanic')
plt.legend(loc='lower right')
plt.grid(True)
plt.axis('equal')
plt.show()

In [ ]:
# Gráfico de barras comparativo
models = ['Logistic Regression', 'Decision Tree', 'Random Forest']
aucs = [auc_log_sk, auc_tree_sk, auc_rf_sk]

plt.figure(figsize=(10, 6))
bars = plt.bar(models, aucs, color=['blue', 'green', 'red'], alpha=0.7)
plt.ylim(0.5, 1.0)
plt.ylabel('AUC')
plt.title('Comparación de AUC por modelo')

# Añadir valores en las barras
for bar, auc_val in zip(bars, aucs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{auc_val:.4f}', ha='center', va='bottom')

plt.grid(True, axis='y')
plt.show()

### 3.1. Interpretación de resultados

*   El **Random Forest** obtiene el AUC más alto, indicando que es el mejor clasificador entre los tres.
*   La **regresión logística** tiene un AUC respetable, demostrando que la relación lineal captura buena parte de la información.
*   El **árbol de decisión** tiene el AUC más bajo, posiblemente por sobreajuste o por su menor capacidad de generalización.

El AUC mide la capacidad del modelo para separar las clases independientemente del umbral. Un valor cercano a 1 indica un excelente clasificador.

---
## Actividad 4: Análisis adicional - Punto de operación óptimo

Podemos elegir el umbral que maximiza, por ejemplo, la métrica F1-score o que minimiza la distancia al punto (0,1).

In [ ]:
# Para Random Forest (el mejor modelo)
fpr_rf_sk, tpr_rf_sk, thresholds_rf = roc_curve(y_test, y_scores_rf)

# Encontrar el umbral que maximiza la distancia a la diagonal (Youden's index)
youden = tpr_rf_sk - fpr_rf_sk
best_idx = np.argmax(youden)
best_thresh = thresholds_rf[best_idx]

print(f"Umbral óptimo según índice de Youden: {best_thresh:.4f}")
print(f"  TPR = {tpr_rf_sk[best_idx]:.4f}, FPR = {fpr_rf_sk[best_idx]:.4f}")

# Visualizamos el punto óptimo en la curva ROC
plt.figure(figsize=(8, 8))
plt.plot(fpr_rf_sk, tpr_rf_sk, 'b-', linewidth=2, label='ROC RF')
plt.plot(fpr_rf_sk[best_idx], tpr_rf_sk[best_idx], 'ro', markersize=10, label='Punto óptimo')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Curva ROC con punto óptimo de operación')
plt.legend()
plt.grid(True)
plt.show()

---
## Ejercicios para el Estudiante

1.  **Nuevo modelo:** Añade un clasificador **KNN** (KNeighborsClassifier) y compara su AUC con los anteriores.

2.  **Validación cruzada:** Utiliza `cross_val_score` con scoring='roc_auc' para obtener una estimación más robusta del AUC para cada modelo. ¿Cambia el orden de rendimiento?

3.  **Curva ROC en entrenamiento:** Dibuja la curva ROC también para el conjunto de entrenamiento. ¿Hay evidencia de overfitting?

4.  **Matriz de confusión:** Para el umbral óptimo encontrado, dibuja la matriz de confusión del Random Forest en test. ¿Qué observas?

5.  **Reflexión:** ¿Por qué el AUC es una métrica preferible a la accuracy en problemas con clases desbalanceadas? ¿Está desbalanceado el dataset Titanic? (Verifica la distribución de `survived`).

In [ ]:
# Verificación de balance de clases
print("Distribución de clases en Titanic:")
print(y.value_counts())
print(f"Proporción de supervivientes: {y.mean():.2%}")

---
## Conclusiones de la Sesión

*   Hemos trabajado con el dataset **Titanic**, preprocesando datos y entrenando tres clasificadores: regresión logística, árbol de decisión y random forest.
*   Calculamos la **curva ROC** manualmente para cada modelo, variando el umbral de clasificación.
*   Implementamos el cálculo del **AUC** mediante integración trapezoidal (`np.trapz`) y verificamos que coincide con `sklearn.metrics.roc_auc_score`.
*   Comparamos los modelos mediante sus curvas ROC y valores de AUC, observando que **Random Forest** supera a los otros.
*   Analizamos el punto de operación óptimo (índice de Youden) para el mejor modelo.
*   Conectamos estos conceptos con la evaluación de modelos en machine learning, destacando la utilidad del AUC como métrica independiente del umbral.

¡Con este ejercicio práctico cerramos el curso, aplicando cálculo integral a una métrica fundamental en ML!